In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 dan Qiskit SDK

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK menyediakan beberapa alat untuk mengonversi antara representasi OpenQASM dari program kuantum dan kelas [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## Impor program OpenQASM 2 ke Qiskit
Ada dua fungsi untuk mengimpor program OpenQASM 2 ke Qiskit.
Fungsi tersebut adalah [`qasm2.load()`](../api/qiskit/qasm2#load), yang menerima nama file, dan [`qasm2.loads()`](../api/qiskit/qasm2#loads), yang menerima program OpenQASM 2 sebagai string.

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Lihat [OpenQASM 2 Qiskit API](https://docs.quantum.ibm.com/api/qiskit/qasm2) untuk informasi lebih lanjut.

### Impor program sederhana
Untuk sebagian besar program OpenQASM 2, kamu bisa langsung menggunakan `qasm2.load` dan `qasm2.loads` dengan satu argumen saja.

#### Contoh: impor program OpenQASM 2 sebagai string
Gunakan `qasm2.loads()` untuk mengimpor program OpenQASM 2 sebagai string ke dalam QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### Contoh: impor program OpenQASM 2 dari file
Gunakan `load()` untuk mengimpor program OpenQASM 2 dari file ke dalam QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### Hubungkan Gate OpenQASM 2 dengan Gate Qiskit
Secara default, importer OpenQASM 2 milik Qiskit memperlakukan file include `"qelib1.inc"` sebagai library standar *de facto*.
Importer memperlakukan file ini sebagai berisi tepat Gate-Gate yang dijelaskan dalam [makalah asli yang mendefinisikan OpenQASM 2](https://arxiv.org/abs/1707.03429).
Qiskit akan menggunakan Gate bawaan dari [circuit library](../api/qiskit/circuit_library) untuk merepresentasikan Gate-Gate dalam `"qelib1.inc"`.
Gate yang didefinisikan dalam program melalui pernyataan `gate` OpenQASM 2 secara manual akan, secara default, dikonstruksi sebagai [subkelas Qiskit `Gate`](../api/qiskit/qiskit.circuit.Gate) khusus.

Kamu bisa memberi tahu importer untuk menggunakan kelas [`Gate`](../api/qiskit/qiskit.circuit.Gate) tertentu untuk pernyataan `gate` yang ditemukan.
Kamu juga bisa menggunakan mekanisme ini untuk memperlakukan nama Gate tambahan sebagai "bawaan", artinya tidak memerlukan definisi eksplisit.
Jika kamu menentukan kelas Gate yang akan digunakan untuk pernyataan `gate` di luar `"qelib1.inc"`, Circuit yang dihasilkan biasanya akan lebih efisien untuk digunakan.

> **Warning:** Mulai Qiskit SDK v1.0, *exporter* OpenQASM 2 milik Qiskit (lihat [Ekspor Circuit Qiskit ke OpenQASM 2](#qasm2-export)) masih berperilaku seolah-olah `"qelib1.inc"` memiliki lebih banyak Gate daripada yang sebenarnya ada.
> Ini berarti pengaturan default importer mungkin tidak bisa mengimpor program yang diekspor oleh exporter kita.
> Lihat [contoh khusus untuk bekerja dengan exporter lama](#qasm2-import-legacy) untuk mengatasi masalah ini.
> 
> Ketidaksesuaian ini adalah perilaku lama Qiskit, dan [akan diselesaikan pada rilis Qiskit berikutnya](https://github.com/Qiskit/qiskit/issues/10737).

Untuk meneruskan informasi tentang instruksi khusus ke importer OpenQASM 2, gunakan [kelas `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
Kelas ini memerlukan empat informasi wajib, secara berurutan:

* **Nama** Gate, yang digunakan dalam program OpenQASM 2
* **Jumlah parameter sudut** yang diterima Gate
* **Jumlah Qubit** yang dioperasikan Gate
* **Kelas konstruktor** Python atau fungsi untuk Gate tersebut, yang menerima parameter Gate (tapi bukan Qubit) sebagai argumen individual

Jika importer menemukan definisi `gate` yang cocok dengan instruksi khusus yang diberikan, importer akan menggunakan informasi khusus tersebut untuk merekonstruksi objek Gate.
Jika pernyataan `gate` ditemukan yang cocok dengan `name` instruksi khusus, tapi tidak cocok dengan jumlah parameter maupun jumlah Qubit, importer akan memunculkan [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror), untuk menunjukkan ketidaksesuaian antara informasi yang diberikan dan program.

Selain itu, argumen kelima `builtin` bisa secara opsional diatur ke `True` untuk membuat Gate tersedia secara otomatis dalam program OpenQASM 2, meskipun tidak didefinisikan secara eksplisit.
Jika importer menemukan definisi `gate` eksplisit untuk instruksi khusus bawaan, importer akan menerimanya secara diam-diam.
Seperti sebelumnya, jika definisi eksplisit dengan nama yang sama tidak kompatibel dengan instruksi khusus yang diberikan, [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) akan dimunculkan.
Ini berguna untuk kompatibilitas dengan exporter OpenQASM 2 yang lebih lama, dan dengan platform kuantum tertentu yang memperlakukan "gate dasar" hardware mereka sebagai instruksi bawaan.

Qiskit menyediakan atribut data untuk bekerja dengan program OpenQASM 2 yang dihasilkan oleh versi lama dari [kemampuan ekspor OpenQASM 2 Qiskit](#qasm2-export).
Atribut ini adalah [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility), yang bisa diberikan sebagai argumen `custom_instructions` ke [`qasm2.load()`](../api/qiskit/qasm2#load) dan [`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### Contoh: impor program yang dibuat oleh exporter lama Qiskit
Program OpenQASM 2 ini menggunakan Gate yang tidak ada dalam versi asli `"qelib1.inc"` tanpa mendeklarasikannya, tapi merupakan Gate standar dalam library Qiskit.
Kamu bisa menggunakan [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) untuk dengan mudah memberi tahu importer agar menggunakan set Gate yang sama yang sebelumnya digunakan oleh exporter OpenQASM 2 Qiskit.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### Contoh: gunakan kelas Gate tertentu saat mengimpor program OpenQASM 2
Qiskit tidak bisa, secara umum, memverifikasi apakah definisi dalam pernyataan `gate` OpenQASM 2 sesuai persis dengan Gate library standar Qiskit.
Sebagai gantinya, Qiskit memilih Gate khusus menggunakan definisi yang tepat yang diberikan.
Ini bisa kurang efisien dibandingkan menggunakan salah satu Gate standar bawaan, atau Gate khusus yang didefinisikan pengguna.
Kamu bisa mendefinisikan pernyataan `gate` secara manual dengan kelas-kelas tertentu.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### Contoh: definisikan Gate bawaan baru dalam program OpenQASM 2
Jika argumen `builtin=True` diatur, Gate khusus tidak perlu memiliki definisi yang terkait.